# Train BarangAI Watch on Kaggle

Compare EfficientNet-B0 and MobileNetV3-Large using the existing project pipeline. This notebook contains no training results.

1. Create a **private** Kaggle notebook and import this file.
2. In Settings select an available free **GPU** accelerator and enable Internet for dependencies/pretrained weights. Do not link paid cloud compute.
3. Add a private Kaggle dataset containing `kaggle-training-code.zip` generated from this repository.
4. Add your reviewed five-class dataset with `manifest.csv` and an `images/` folder. Raw public datasets need image-rights, labels and incident grouping reviewed first. Do not upload private resident photos or consent records without appropriate permission.
5. Adjust the dataset path below, then Run All. Keep the notebook and outputs private until reviewed.


In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, shutil, json
DATASET_DIR = Path('/kaggle/input/barangai-five-class')
MANIFEST = DATASET_DIR / 'manifest.csv'
IMAGE_ROOT = DATASET_DIR / 'images'
WORK = Path('/kaggle/working/barangai-ml')
OUTPUT = WORK / 'outputs' / 'run1'
EPOCHS = 10
BATCH_SIZE = 16
SEED = 42
assert MANIFEST.is_file(), 'Attach the reviewed dataset and update DATASET_DIR.'
assert IMAGE_ROOT.is_dir(), 'Dataset must contain images/.'


## Load the project training code

In [ ]:
archives = list(Path('/kaggle/input').rglob('kaggle-training-code.zip'))
assert len(archives) == 1, 'Attach exactly one project training-code archive.'
WORK.mkdir(parents=True, exist_ok=True)
expected = {'barangai_ml/__init__.py', 'barangai_ml/data.py', 'barangai_ml/cli.py', 'requirements.txt'}
with zipfile.ZipFile(archives[0]) as bundle:
    assert set(bundle.namelist()) == expected, 'Unexpected training archive contents.'
    for name in expected:
        target = WORK / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(bundle.read(name))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WORK / 'requirements.txt')], check=True)
import torch
print('Torch:', torch.__version__, 'GPU:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a free GPU accelerator before starting training.'


## Validate provenance and freeze the split

The pipeline rejects missing classes, missing provenance, hash mismatches and incident/source leakage. Keep the same split for both models. Review visually similar photos manually too.

In [ ]:
SPLIT = WORK / 'outputs' / 'split.csv'
subprocess.run([sys.executable, '-m', 'barangai_ml.cli', 'prepare',
    '--manifest', str(MANIFEST), '--image-root', str(IMAGE_ROOT),
    '--output', str(SPLIT), '--seed', str(SEED)], cwd=WORK, check=True)


## Fine-tune both pretrained models

The existing command trains both architectures sequentially. It selects checkpoints and the candidate using validation macro F1. Ten epochs are a starting experiment, not a promised sufficient training duration. If GPU memory is exhausted, reduce batch size and document the change consistently for both models.

In [ ]:
subprocess.run([sys.executable, '-m', 'barangai_ml.cli', 'train',
    '--split', str(SPLIT), '--image-root', str(IMAGE_ROOT), '--output', str(OUTPUT),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--seed', str(SEED)],
    cwd=WORK, check=True)
print((OUTPUT / 'results.json').read_text())


## Export and review

Save a notebook version with outputs. Download the archive below. Review class errors, validation confidence/coverage, checkpoints and CPU latency. Select the confidence threshold using validation predictions only; do not tune on test predictions. Rebenchmark the selected model on the actual host before enabling it. A Kaggle GPU training success does not prove the free web host has enough memory.

In [ ]:
archive = shutil.make_archive('/kaggle/working/barangai-training-results', 'zip', WORK / 'outputs')
print('Download:', archive)
